In [ ]:
# Scenario Question
# "Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then either publishes or rejects them. How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"

In [1]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List, Dict
import requests
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

# 1. State Definition
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool
    reviewer: str
    feedback: str
    status: str
    history: List[Dict]


# 2. NVIDIA API Call
def nvidia_call(prompt: str):
    api_key = os.getenv("NVIDIA_API_KEY")

    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta/llama3-8b-instruct",
            "messages": [
                {"role": "system", "content": "You are a professional report writing assistant."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.5
        }
    )

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()["choices"][0]["message"]["content"]


# 3. Draft Node
def draft_report(state: ReportState):
    print(f"\n📝 Generating report for: {state['task']}")

    draft = nvidia_call(f"""
Create a structured professional report for: {state['task']}

Include:
- Title
- Introduction
- Key Points
- Conclusion
""")

    return {
        "draft": draft,
        "status": "DRAFTED"
    }


# 4. Review Node (Interrupt)
def review_node(state: ReportState):
    print("\n📄 Draft for Review:\n")
    print(state["draft"])
    print("\n👉 Approve? (yes/no)")
    print("👉 Optional feedback:")

    return {}  # pause


# 5. Decision Node
def decision_node(state: ReportState):
    if state["approved"]:
        return "publish"
    else:
        return "reject"


# 6. Publish Node
def publish_report(state: ReportState):
    print("\n✅ Report Approved & Published")

    entry = {
        "action": "PUBLISHED",
        "reviewer": state["reviewer"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "status": "PUBLISHED",
        "history": state["history"] + [entry]
    }


# 7. Reject Node
def reject_report(state: ReportState):
    print("\n❌ Report Rejected")

    entry = {
        "action": "REJECTED",
        "reviewer": state["reviewer"],
        "feedback": state["feedback"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "status": "REJECTED",
        "history": state["history"] + [entry]
    }


# 8. Build Graph
graph = StateGraph(ReportState)

graph.add_node("draft", draft_report)
graph.add_node("review", review_node)
graph.add_node("publish", publish_report)
graph.add_node("reject", reject_report)

graph.set_entry_point("draft")
graph.add_edge("draft", "review")

graph.add_conditional_edges(
    "review",
    decision_node,
    {
        "publish": "publish",
        "reject": "reject"
    }
)

graph.add_edge("publish", END)
graph.add_edge("reject", END)


# 9. Compile with Checkpointer
checkpointer = MemorySaver()

app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)


# 10. Run
if __name__ == "__main__":
    thread = {"configurable": {"thread_id": "report-1"}}

    # Step 1: Generate draft
    app.invoke({
        "task": "Q3 Financial Report",
        "draft": "",
        "approved": False,
        "reviewer": "",
        "feedback": "",
        "status": "",
        "history": []
    }, thread)

    # Step 2: Human Review
    decision = input("\nApprove? (yes/no): ").strip().lower()
    feedback = input("Feedback (optional): ")
    reviewer = input("Reviewer name: ")

    approved = True if decision == "yes" else False

    # Step 3: Resume
    result = app.invoke({
        "approved": approved,
        "feedback": feedback,
        "reviewer": reviewer
    }, thread)

    # Final Output
    print("\n📊 FINAL STATE:")
    print(result)


📝 Generating report for: Q3 Financial Report

📝 Generating report for: Q3 Financial Report

📊 FINAL STATE:
{'task': 'Q3 Financial Report', 'draft': "**Q3 Financial Report**\n\n**Title:** Quarterly Financial Report for the Third Quarter of [Year]\n\n**Introduction:**\n\nThis report provides an overview of the financial performance of [Company Name] for the third quarter of [Year]. The report summarizes the key financial metrics, highlights notable trends, and provides insights into the company's financial position and performance.\n\n**Key Points:**\n\n**Revenue:**\n\n* Total revenue for Q3 [Year] was $[X] million, representing a [Y]% increase from Q3 [Previous Year].\n* Revenue from [Segment/Division] increased by [Z]% to $[W] million, driven by strong demand and market growth.\n* Revenue from [Segment/Division] decreased by [U]% to $[V] million, due to market fluctuations and competition.\n\n**Expenses:**\n\n* Total operating expenses for Q3 [Year] were $[X] million, representing a [